In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix

from importlib import reload
import prepare_data as pda
reload(pda)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dhaaa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dhaaa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dhaaa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dhaaa\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


<module 'prepare_data' from 'c:\\Users\\dhaaa\\OneDrive\\Skrivebord\\Skole\\Gruppe INFO284\\Info284\\prepare_data.py'>

In [3]:
data = pd.read_csv("./data/Hotel_Reviews.csv")
X, y = pda.prepare_lstm(data)

In [4]:
# Tokenize the data
tokenizer = Tokenizer(num_words = 20000)
tokenizer.fit_on_texts(X)
sequences = tokenizer.texts_to_sequences(X)

# Pad the sequences
maxlen = 52
data = pad_sequences(sequences, maxlen = maxlen)

# Convert labels to numpy array
labels = np.array(y, dtype='int')

print(data.shape)
print(labels.shape)

(515738, 52)
(515738, 3)


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

print(y_val.shape)
print(y_train.shape)

(103148, 3)
(412590, 3)


In [1]:
model = Sequential()
model.add(Embedding(input_dim = 20000, output_dim = 128, mask_zero=True))
model.add(LSTM(units = 128, return_sequences = False))
model.add(Dropout(0.5))
model.add(Dense(20, input_shape=52, kernel_initializer='he_uniform', activation='relu'))
model.add(Dropout(0.3, noise_shape=None))
model.add(Dense(3))
model.compile(loss='mae', optimizer='adam', metrics = ['accuracy'])

NameError: name 'Sequential' is not defined

In [ ]:
# Train the model
model.fit(X_train, y_train, epochs = 5, batch_size = 20000, validation_data=(X_val, y_val))

# Save the model in the recommended Keras format
model.save('./data/model/sentiment_lstm_model.keras')

print("Model training complete and saved as 'data/model/sentiment_lstm_model.keras'")

Epoch 1/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 100s 5s/step - accuracy: 0.6021 - loss: 0.3117 - val_accuracy: 0.6936 - val_loss: 0.2330
Epoch 2/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 90s 4s/step - accuracy: 0.7178 - loss: 0.2396 - val_accuracy: 0.7630 - val_loss: 0.2020
Epoch 3/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 90s 4s/step - accuracy: 0.7666 - loss: 0.2108 - val_accuracy: 0.7706 - val_loss: 0.1885
Epoch 4/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 90s 4s/step - accuracy: 0.7765 - loss: 0.1978 - val_accuracy: 0.7756 - val_loss: 0.1774
Epoch 5/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 90s 4s/step - accuracy: 0.7831 - loss: 0.1862 - val_accuracy: 0.7778 - val_loss: 0.1693
Model training complete and saved as 'data/model/sentiment_lstm_model.keras'


In [ ]:
# Load the model
model = load_model('./data/model/sentiment_lstm_model.keras')

# Evaluate the model on the validation set
val_predictions = model.predict(X_val)
val_predictions = np.argmax(val_predictions, axis=1)
y_val = np.argmax(y_val, axis=1)

# Print classification report and confusion matrix
print(classification_report(y_val, val_predictions))
print(confusion_matrix(y_val, val_predictions))

3224/3224 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      2165
           1       0.69      0.61      0.65     34000
           2       0.81      0.89      0.85     66983

    accuracy                           0.78    103148
   macro avg       0.50      0.50      0.50    103148
weighted avg       0.76      0.78      0.76    103148

[[    0  1773   392]
 [    0 20589 13411]
 [    0  7347 59636]]


c:\Users\dhaaa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\dhaaa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\dhaaa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Select a subset of the data for prediction
sample_reviews = X_val[:10]  # Select the first 10 reviews for prediction
sample_labels = y_val[:10]  # Corresponding labels

# Make predictions
predictions = model.predict(sample_reviews)

# Print predictions
for review, prediction, label in zip(sample_reviews, predictions, sample_labels):
    print(prediction)
    pred = max(prediction)
    index = 0
    for n in range(len(prediction)):
        if prediction[n] == pred:
            index = n
    if index == 0:
        sentiment = "Negative"
    elif index == 1:
        sentiment = "Neutral"
    else:
        sentiment = "Positive"
    if label == 2:
        actual_sentiment = "Positive"
    elif label == 1:
        actual_sentiment = "Neutral"
    else:
        actual_sentiment = "Negative"
    print(f"Review: {review} - Predicted Sentiment: {sentiment} - Actual Sentiment: {actual_sentiment}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
[-0.00163869  0.0054082   1.0022153 ]
Review: [  27  867  222   26  863  126 2741  247  264  245  650  160  244  173
   48    3    8   13   10    2   54    5  125  115    6  283    6   66
  445   12  635    1  148  333    1 1706  114  396    3   62  149  836
   65  133   30  554   44  247   62   17  131   18] - Predicted Sentiment: Positive - Actual Sentiment: Positive
[0.00564763 0.78593767 0.0076748 ]
Review: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0 1722  787  818 1556    1   31] - Predicted Sentiment: Neutral - Actual Sentiment: Positive
[-0.00402744  0.01081684  0.78278863]
Review: [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0  